# Sparse-Filtered LGBM With Row-Wise Features

Test row-wise aggregate features after applying the CV-selected sparse threshold from `05_sparsity_thresholds_lgbm.ipynb`.

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd

In [2]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, root_mean_squared_log_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split

from src.loader import Loader
from src.modeling import build_lgbm_regressor
from src.preprocessing import FeaturePreprocessor

In [3]:
SEED = 42
TEST_SIZE = 0.33
CV = 5
SPARSITY_THRESHOLD = 0.9875

In [4]:
loader = Loader()
df = loader.load("../data/processed_data.csv")
df.shape

(4459, 4732)

In [5]:
X = df.drop(columns="target")
y = df["target"]
y_log = np.log1p(y)

(X.shape, y.shape)

((4459, 4731), (4459,))

The sparse threshold is selected from notebook `05`. The row-wise features are added after sparse filtering so this notebook tests the combined setup.

In [6]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw, y_train_log, y_test_log = train_test_split(
    X,
    y,
    y_log,
    test_size=TEST_SIZE,
    random_state=SEED,
)

preprocessor = FeaturePreprocessor(
    zero_share_threshold=SPARSITY_THRESHOLD,
    add_rowwise=True,
)
X_aug_train = preprocessor.fit_transform(X_train_raw)
X_aug_test = preprocessor.transform(X_test_raw)

cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

pd.DataFrame(
    {
        "metric": ["sparsity_threshold", "base_features_kept", "added_rowwise_features", "final_features"],
        "value": [
            SPARSITY_THRESHOLD,
            len(preprocessor.columns_to_keep_),
            X_aug_train.shape[1] - len(preprocessor.columns_to_keep_),
            X_aug_train.shape[1],
        ],
    }
)

,metric,value
0,sparsity_threshold,0.9875
1,base_features_kept,2482.0000
2,added_rowwise_features,8.0000
3,final_features,2490.0000


In [7]:
cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

In [8]:
default_params = {
    "n_estimators": 500,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": 8,
    "min_child_samples": 20,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.05,
    "reg_lambda": 0.05,
    "min_split_gain": 0.0,
}

best_params = default_params

In [9]:
model = build_lgbm_regressor(best_params)
cv_scores = -cross_val_score(
    estimator=model,
    X=X_aug_train,
    y=y_train_log,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=1,
)

In [10]:
model.fit(X_aug_train, y_train_log)
y_pred_log = model.predict(X_aug_test)
y_pred = np.expm1(y_pred_log)
y_pred = np.clip(y_pred, 0, None)

results_df = pd.DataFrame(
    {
        "metric": [
            "sparsity_threshold",
            "base_features_kept",
            "added_rowwise_features",
            "n_features",
            "cv_rmsle_mean",
            "cv_rmsle_std",
            "test_rmsle",
            "test_rmse",
            "test_mae",
            "test_r2",
        ],
        "value": [
            SPARSITY_THRESHOLD,
            len(preprocessor.columns_to_keep_),
            X_aug_train.shape[1] - len(preprocessor.columns_to_keep_),
            X_aug_train.shape[1],
            cv_scores.mean(),
            cv_scores.std(),
            root_mean_squared_log_error(y_test_raw, y_pred),
            root_mean_squared_error(y_test_raw, y_pred),
            mean_absolute_error(y_test_raw, y_pred),
            r2_score(y_test_raw, y_pred),
        ],
    }
)

results_df.style.format({"value": "{:,.4f}"})

,metric,value
0,sparsity_threshold,0.9875
1,base_features_kept,"2,482.0000"
2,added_rowwise_features,8.0000
3,n_features,"2,490.0000"
4,cv_rmsle_mean,1.3757
5,cv_rmsle_std,0.0223
6,test_rmsle,1.3851
7,test_rmse,"6,886,340.3072"
8,test_mae,"3,863,837.6481"
9,test_r2,0.2570


In [11]:
added_features = [
    col for col in X_aug_train.columns
    if col not in preprocessor.columns_to_keep_
]
added_features

['non_zero_count',
 'non_zero_ratio',
 'row_sum',
 'row_mean',
 'row_std',
 'row_max',
 'nz_mean',
 'nz_std']

In [12]:
summary = {
    "target_transform": "log1p",
    "primary_metric": "rmsle",
    "sparsity_threshold": SPARSITY_THRESHOLD,
    "base_features_kept": len(preprocessor.columns_to_keep_),
    "model_params": best_params,
    "added_features": added_features,
    "results": dict(zip(results_df["metric"], results_df["value"])),
}

summary

{'target_transform': 'log1p',
 'primary_metric': 'rmsle',
 'sparsity_threshold': 0.9875,
 'base_features_kept': 2482,
 'model_params': {'n_estimators': 500,
  'learning_rate': 0.03,
  'num_leaves': 31,
  'max_depth': 8,
  'min_child_samples': 20,
  'subsample': 0.8,
  'subsample_freq': 1,
  'colsample_bytree': 0.8,
  'reg_alpha': 0.05,
  'reg_lambda': 0.05,
  'min_split_gain': 0.0},
 'added_features': ['non_zero_count',
  'non_zero_ratio',
  'row_sum',
  'row_mean',
  'row_std',
  'row_max',
  'nz_mean',
  'nz_std'],
 'results': {'sparsity_threshold': 0.9875,
  'base_features_kept': 2482.0,
  'added_rowwise_features': 8.0,
  'n_features': 2490.0,
  'cv_rmsle_mean': 1.3756589695205137,
  'cv_rmsle_std': 0.022255684385063847,
  'test_rmsle': 1.3851137689136672,
  'test_rmse': 6886340.307222505,
  'test_mae': 3863837.6481209593,
  'test_r2': 0.25699309315198726}}

## How To Read The Result

- Compare the `cv_rmsle_mean` and test `RMSLE` here against notebook `05`, which tested sparse filtering without row-wise features.
- This notebook uses the CV-selected sparse threshold `0.9875` from notebook `05`.
- Accept the row-wise features only if they improve the sparse-filtered setup without a material test regression.
- If this wins, the next step is to rerun `Optuna` on the sparse-filtered augmented feature space.

# 06 Add Row-Wise Features Report

## Goal

The goal of this notebook was to test row-wise aggregate features after applying the CV-selected sparse threshold from notebook `05`.

## What Was Done

- Loaded `data/processed_data.csv`.
- Split the data into train and test parts.
- Applied sparse filtering with threshold `0.9875`, selected from `05_sparsity_thresholds_lgbm.ipynb` by CV RMSLE.
- Kept `2,482` base features after sparse filtering.
- Added row-wise features:
  - `non_zero_count`
  - `non_zero_ratio`
  - `row_sum`
  - `row_mean`
  - `row_std`
  - `row_max`
  - `nz_mean`
  - `nz_std`
- Trained LightGBM with fixed tuned parameters on `log1p(target)`.
- Evaluated CV RMSLE and held-out test metrics.

## Main Results

| metric | value |
| --- | ---: |
| Sparse threshold | 0.9875 |
| Base features kept | 2,482 |
| Added row-wise features | 8 |
| Final feature count | 2,490 |
| CV RMSLE mean | 1.3757 |
| CV RMSLE std | 0.0223 |
| Test RMSLE | 1.3851 |
| Test RMSE | 6,886,340.31 |
| Test MAE | 3,863,837.65 |
| Test R2 | 0.2570 |

## Comparison To Notebook 05

Notebook `05` tested sparse filtering without row-wise features. Its CV-selected threshold `0.9875` had CV RMSLE `1.4500` and test RMSLE `1.4635`.

After adding row-wise features on top of the same sparse-filtered base features, CV RMSLE improved to `1.3757` and test RMSLE improved to `1.3851`.

## Conclusion

Row-wise aggregate features help substantially after sparse filtering. This combined setup is stronger than sparse filtering alone and should be the feature setup used before final Optuna tuning.
